In [11]:

import os, glob
from omegaconf import OmegaConf
# import pretty print
from pprint import pprint


In [14]:
# get all the result yaml files and parse them with omegaconf

def parse_results(file_path):
    with open(file_path, 'r') as f:
        conf = OmegaConf.load(f)
    return conf
tasks = ['location','age','bmi','sex','supplement','helicobacter_pylori_infection','hbv_infection']
reg_tasks = ['age','bmi']
class_tasks = [t for t in tasks if t not in reg_tasks]
best_res_mae = {t:{} for t in reg_tasks}
best_res_r2 = {t:{} for t in reg_tasks}
best_res_weighted_auroc = {t:{} for t in class_tasks}
best_res_accuracy = {t:{} for t in class_tasks}

topdir = '/project/aip-rahulgk/dpellow/gut_microbiome_GPT/outputs/finetune/'
# get list of result files:
for task in tasks:
    file_regex = f'{topdir}/{task}/*/best_model/test_metrics.yaml'
    res_files = glob.glob(file_regex)
    # get the top five results for each metric and store them in a dict
    
    for res_file in res_files:
        res = parse_results(res_file)
        # get the parent dir name which is the model name
        model_name = os.path.basename(os.path.dirname(os.path.dirname(res_file)))
        if task in reg_tasks:
            mae = res['test_mae']
            r2 = res['test_r2']
            best_res_mae[task][model_name] = mae
            best_res_r2[task][model_name] = r2
        else:
            best_res_weighted_auroc[task][model_name] = res['test_auroc_weighted'] if 'test_auroc_weighted' in res else res['test_auroc']
            best_res_accuracy[task][model_name] = res['test_accuracy']
    # keep the top five results for each metric
    print(80*'=')
    print(f'Top 5 results for {task}:')
    if task in reg_tasks:
        best_res_mae[task] = dict(sorted(best_res_mae[task].items(), key=lambda item: item[1])[:5])
        best_res_r2[task] = dict(sorted(best_res_r2[task].items(), key=lambda item: item[1], reverse=True)[:5])
        print(f'MAE:')
        pprint(sorted(best_res_mae[task].items(), key=lambda item: item[1]))
        # pprint(best_res_mae[task])
        print(f'R2:')
        # pprint(best_res_r2[task])
        pprint(sorted(best_res_r2[task].items(), key=lambda item: item[1], reverse=True))
    else:
        best_res_weighted_auroc[task] = dict(sorted(best_res_weighted_auroc[task].items(), key=lambda item: item[1], reverse=True)[:5])
        best_res_accuracy[task] = dict(sorted(best_res_accuracy[task].items(), key=lambda item: item[1], reverse=True)[:5])
        print('Weighted AUROC:')
        pprint(sorted(best_res_weighted_auroc[task].items(), key=lambda item: item[1], reverse=True))
        # pprint(best_res_weighted_auroc[task])
        print(f'Accuracy:')
        pprint(sorted(best_res_accuracy[task].items(), key=lambda item: item[1], reverse=True))
        # pprint(best_res_accuracy[task])



Top 5 results for location:
Weighted AUROC:
[('masking_log_rel_ab_len200_3layers_ratio_70_updated_cls', 0.9480065946939529),
 ('masking_clr_len200_3layers_ratio_70_lr001_updated_cls', 0.9304291975800826),
 ('masking_log_rel_ab_len200_3layers_ratio_70_updated_pool',
  0.9290952955011005),
 ('masking_log_rel_ab_len200_3layers_ratio_50_updated_pool',
  0.9264474083805936),
 ('masking_log_rel_ab_len200_3layers_ratio_30_updated_cls', 0.9235336075183732)]
Accuracy:
[('masking_log_rel_ab_len200_3layers_ratio_30_updated_cls', 0.8103130755064457),
 ('masking_log_rel_ab_len200_3layers_ratio_50_updated_pool',
  0.8029465930018416),
 ('masking_log_rel_ab_len200_3layers_ratio_30_updated_pool',
  0.8001841620626151),
 ('masking_log_rel_ab_len200_3layers_ratio_70_updated_cls', 0.7914364640883977),
 ('masking_clr_len200_3layers_ratio_50_lr001_updated_cls', 0.7863720073664825)]
Top 5 results for age:
MAE:
[('masking_MSE_LOSS_log_rel_ab_len200_3layers_ratio_50_updated_pool',
  24.219650268554688),
 ('ma